In [ ]:
%pip install pennylane torch torchvision matplotlib scikit-learn
%pip install medmnist

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 36.2 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.7 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from medmnist import PneumoniaMNIST

# Transform grayscale (1 channel) to 3-channel RGB and resize for ResNet-18
data_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Download and load official PneumoniaMNIST splits
train_raw = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_raw   = PneumoniaMNIST(split='val',   transform=data_transform, download=True)
test_raw  = PneumoniaMNIST(split='test',  transform=data_transform, download=True)

print(f"Dataset loaded: {len(train_raw)} train, {len(val_raw)} val, {len(test_raw)} test images.")

100%|██████████| 4.17M/4.17M [00:07<00:00, 589kB/s]


Dataset loaded: 4708 train, 524 val, 624 test images.


In [ ]:
from torchvision import models
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained ResNet-18 and strip the final classification head
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(device)
feature_extractor.eval()

def extract_features(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_features = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Extracting Features"):
            imgs = imgs.to(device)
            feats = feature_extractor(imgs)          # Shape: (B, 512, 1, 1)
            feats = torch.flatten(feats, 1)          # Shape: (B, 512)
            all_features.append(feats.cpu())
            all_labels.append(lbls.squeeze().long()) # Shape: (B,)

    return torch.cat(all_features), torch.cat(all_labels)

# Extract and create fast TensorDataLoaders
X_train, y_train = extract_features(train_raw)
X_val,   y_val   = extract_features(val_raw)
X_test,  y_test  = extract_features(test_raw)

batch_size = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
print(f"Extracted feature shape: {X_train.shape}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 116MB/s]
Extracting Features: 100%|██████████| 10/10 [01:14<00:00,  7.49s/it]

Extracted feature shape: torch.Size([4708, 512])


In [ ]:
import copy
import torch.optim as optim

def train_and_evaluate(model, dataloaders, epochs=15, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            total_loss, correct, total = 0.0, 0, 0

            for feats, labels in dataloaders[phase]:
                feats, labels = feats.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(feats)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                total_loss += loss.item() * feats.size(0)
                correct += torch.sum(preds == labels).item()
                total += labels.size(0)

            epoch_loss = total_loss / total
            epoch_acc = correct / total
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Acc: {history['train_acc'][-1]:.4f} | Val Acc: {history['val_acc'][-1]:.4f}")

    model.load_state_dict(best_weights)
    return model, history

In [ ]:
class BottleneckClassicalClassifier(nn.Module):
    def __init__(self, in_features=512, hidden_dim=4, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

classical_model = BottleneckClassicalClassifier().to(device)
print("Training Classical Baseline Model...")
classical_model, classical_history = train_and_evaluate(classical_model, dataloaders, epochs=15, lr=1e-3)

Training Classical Baseline Model...
Epoch 01/15 | Train Acc: 0.8645 | Val Acc: 0.9046
Epoch 02/15 | Train Acc: 0.9327 | Val Acc: 0.9198
Epoch 03/15 | Train Acc: 0.9395 | Val Acc: 0.9351
Epoch 04/15 | Train Acc: 0.9503 | Val Acc: 0.9466
Epoch 05/15 | Train Acc: 0.9494 | Val Acc: 0.9466
Epoch 06/15 | Train Acc: 0.9554 | Val Acc: 0.9504
Epoch 07/15 | Train Acc: 0.9552 | Val Acc: 0.9504
Epoch 08/15 | Train Acc: 0.9541 | Val Acc: 0.9542
Epoch 09/15 | Train Acc: 0.9560 | Val Acc: 0.9561
Epoch 10/15 | Train Acc: 0.9645 | Val Acc: 0.9504
Epoch 11/15 | Train Acc: 0.9650 | Val Acc: 0.9504
Epoch 12/15 | Train Acc: 0.9639 | Val Acc: 0.9561
Epoch 13/15 | Train Acc: 0.9656 | Val Acc: 0.9504
Epoch 14/15 | Train Acc: 0.9662 | Val Acc: 0.9599
Epoch 15/15 | Train Acc: 0.9660 | Val Acc: 0.9580


In [ ]:
import pennylane as qml
import numpy as np
import torch
import torch.nn as nn

n_qubits = 4
q_depth = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # AngleEmbedding natively supports batched inputs (batch_size, n_qubits)
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")

    # Variational layers: single-qubit rotations followed by entangling CNOTs
    for l in range(q_depth):
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])

    # Measure Pauli-Z expectation values across all 4 qubits
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (q_depth, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

class HybridQuantumClassifier(nn.Module):
    def __init__(self, in_features=512, num_classes=2):
        super().__init__()
        self.pre_net = nn.Sequential(
            nn.Linear(in_features, n_qubits),
            nn.Tanh()
        )
        self.qlayer = qlayer
        self.post_net = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        # Scale tanh outputs into [-pi/2, pi/2]
        x = self.pre_net(x) * (np.pi / 2.0)

        # TorchLayer automatically broadcasts over batch dimension
        x = self.qlayer(x)

        # Ensure tensor matches post_net's device and dtype
        x = x.to(device=self.post_net.weight.device, dtype=torch.float32)
        x = self.post_net(x)
        return x

# Sample 4-element dummy input and matching weight shape
dummy_inputs = torch.zeros(4)
dummy_weights = torch.zeros((q_depth, n_qubits, 3))

# Print standard ASCII diagram
print(qml.draw(quantum_circuit)(dummy_inputs, dummy_weights))

hybrid_model = HybridQuantumClassifier().to(device)
print("Training Hybrid Quantum Model...")
hybrid_model, quantum_history = train_and_evaluate(hybrid_model, dataloaders, epochs=15, lr=5e-3)

0: ─╭AngleEmbedding(M0)──Rot(0.00,0.00,0.00)─╭●───────╭X──Rot(0.00,0.00,0.00)─╭●───────╭X─┤  <Z>
1: ─├AngleEmbedding(M0)──Rot(0.00,0.00,0.00)─╰X─╭●────│───Rot(0.00,0.00,0.00)─╰X─╭●────│──┤  <Z>
2: ─├AngleEmbedding(M0)──Rot(0.00,0.00,0.00)────╰X─╭●─│───Rot(0.00,0.00,0.00)────╰X─╭●─│──┤  <Z>
3: ─╰AngleEmbedding(M0)──Rot(0.00,0.00,0.00)───────╰X─╰●──Rot(0.00,0.00,0.00)───────╰X─╰●─┤  <Z>

M0 = 
tensor([0., 0., 0., 0.])
Training Hybrid Quantum Model...
Epoch 01/15 | Train Acc: 0.8704 | Val Acc: 0.8282
Epoch 02/15 | Train Acc: 0.9199 | Val Acc: 0.9485
Epoch 03/15 | Train Acc: 0.9422 | Val Acc: 0.9084
Epoch 04/15 | Train Acc: 0.9373 | Val Acc: 0.9447
Epoch 05/15 | Train Acc: 0.9482 | Val Acc: 0.9447
Epoch 06/15 | Train Acc: 0.9499 | Val Acc: 0.9504
Epoch 07/15 | Train Acc: 0.9539 | Val Acc: 0.8721
Epoch 08/15 | Train Acc: 0.9524 | Val Acc: 0.9408
Epoch 09/15 | Train Acc: 0.9562 | Val Acc: 0.9618
Epoch 10/15 | Train Acc: 0.9571 | Val Acc: 0.9561
Epoch 11/15 | Train Acc: 0.9592 | Val Acc: 0.95